In [1]:
"""
Build the MONTHLY time series for Exit Intention vs. CVE crisis analysis.

Reads the already-built exit_daily_series.csv (from build_q2_timeseries.py /
q2_timeseris.ipynb) -- does NOT re-touch the raw exit_annotated_pass1.csv or
nvd_merged_full.csv files. This mirrors the daily->weekly resample step in
the original script, just resampled to calendar months instead of weeks.

Columns in output (same set as daily/weekly):
  n_exit_any        count, broader exit-intention threshold
  n_exit_explicit   count, stricter exit-intention threshold
  n_total_posts     total burnout posts that month (the local denominator)
  exit_rate_any            n_exit_any / n_total_posts
  exit_rate_explicit       n_exit_explicit / n_total_posts
  n_cve_run1        CVEs that month with cvss_base_score > 9.5
  n_cve_run2        CVEs that month with cvss_base_score >= 10

Rates are recomputed from summed counts (NOT averaged from daily rates),
same reasoning as the weekly step: summing counts first and dividing once
is correct when daily post volume varies.

Output:
  exit_monthly_series.csv   one row per calendar month, 2018-01 to present
"""

import pandas as pd
import numpy as np
import os

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q2_analysis_v2/"
DAILY_CSV = os.path.join(OUT_DIR, "exit_daily_series.csv")

COUNT_COLS = ["n_exit_any", "n_exit_explicit", "n_total_posts", "n_cve_run1", "n_cve_run2"]


def main():
    if not os.path.exists(DAILY_CSV):
        print(f"Could not find {DAILY_CSV}. Run build_q2_timeseries.py (q2_timeseris.ipynb) first.")
        return

    daily = pd.read_csv(DAILY_CSV)
    daily["date"] = pd.to_datetime(daily["date"])
    print(f"Loaded {len(daily)} days from {DAILY_CSV}")

    monthly = daily.set_index("date")[COUNT_COLS].resample("MS").sum().reset_index()

    monthly["exit_rate_any"] = monthly["n_exit_any"] / monthly["n_total_posts"].replace(0, np.nan)
    monthly["exit_rate_explicit"] = monthly["n_exit_explicit"] / monthly["n_total_posts"].replace(0, np.nan)

    n_zero_post_months = (monthly["n_total_posts"] == 0).sum()
    print(f"\nFull monthly series: {len(monthly)} months, "
          f"{monthly['date'].min().strftime('%Y-%m')} to {monthly['date'].max().strftime('%Y-%m')}")
    print(f"Months with zero burnout posts (rate undefined, will be NaN): {n_zero_post_months}")

    out_path = os.path.join(OUT_DIR, "exit_monthly_series.csv")
    monthly.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")

    print("\nDone. Point granger_q2_monthly.py at exit_monthly_series.csv next.")


if __name__ == "__main__":
    main()

Loaded 3071 days from /Users/nadia/Desktop/redditRun_june/q2_analysis_v2/exit_daily_series.csv

Full monthly series: 101 months, 2018-01 to 2026-05
Months with zero burnout posts (rate undefined, will be NaN): 1
Saved -> /Users/nadia/Desktop/redditRun_june/q2_analysis_v2/exit_monthly_series.csv

Done. Point granger_q2_monthly.py at exit_monthly_series.csv next.
